In [11]:
import pandas as pd
import requests
import time
import shutil

import yaml
from pathlib import Path

In [2]:
config_path = Path.cwd().parent / "config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

root = Path(config['project_root'])

In [28]:
# Get weather data
stations = {
    "ALGORTA_BBIZI2": (43.362056, -3.022782),
    "BARAKALDO": (43.298379, -2.987133),
    "BASAURI": (43.241131, -2.883761),
    "ERANDIO": (43.302653, -2.977240),
    "MAZARREDO": (43.267506, -2.935188),
    "MUSKIZ": (43.320713, -3.112716),
    "SANTURCE": (43.333012, -3.042560)
}

for name, (lat, lon) in stations.items():
    print(f"Processing: {name}...")
    
    url = (
        "https://archive-api.open-meteo.com/v1/era5"
        f"?latitude={lat}&longitude={lon}"
        "&start_date=2015-01-01&end_date=2026-05-06"
        "&daily=temperature_2m_mean,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant"
        "&timezone=Europe/Madrid"
    )
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        
        if "daily" in data and "time" in data["daily"]:
            df = pd.DataFrame(data["daily"])
            
            df.rename(columns={
                "temperature_2m_mean": "Temperature",
                "relative_humidity_2m_mean": "Humidity",
                "precipitation_sum": "Precipitation",
                "wind_speed_10m_max": "WindSpeed",
                "wind_direction_10m_dominant": "WindDirection"
            }, inplace=True)
            
            df["Date"] = pd.to_datetime(df["time"])
            df.drop(columns=["time"], inplace=True)
            
            df.to_csv(root / "data" / "raw" / f"{name}_weather.csv", index=False)
            print(f"✅ Success: {name} saved.")
        else:
            print(f"❌ Error: Could not extract data for {name}.")
            
    except Exception as e:
        print(f"⚠️ Error for {name}: {e}")
    
    time.sleep(1.5) 

print("\nFinished!")

Processing: ALGORTA_BBIZI2...
✅ Success: ALGORTA_BBIZI2 saved.
Processing: BARAKALDO...
✅ Success: BARAKALDO saved.
Processing: BASAURI...
✅ Success: BASAURI saved.
Processing: ERANDIO...
✅ Success: ERANDIO saved.
Processing: MAZARREDO...
✅ Success: MAZARREDO saved.
Processing: MUSKIZ...
❌ Error: Could not extract data for MUSKIZ.
Processing: SANTURCE...
❌ Error: Could not extract data for SANTURCE.

Finished!


In [30]:
# Get weather data
stations = {

    "MUSKIZ": (43.320713, -3.112716),
    "SANTURCE": (43.333012, -3.042560)

}

for name, (lat, lon) in stations.items():
    print(f"Processing: {name}...")
    
    url = (
        "https://archive-api.open-meteo.com/v1/era5"
        f"?latitude={lat}&longitude={lon}"
        "&start_date=2015-01-01&end_date=2026-05-06"
        "&daily=temperature_2m_mean,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant"
        "&timezone=Europe/Madrid"
    )
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        
        if "daily" in data and "time" in data["daily"]:
            df = pd.DataFrame(data["daily"])
            
            df.rename(columns={
                "temperature_2m_mean": "Temperature",
                "relative_humidity_2m_mean": "Humidity",
                "precipitation_sum": "Precipitation",
                "wind_speed_10m_max": "WindSpeed",
                "wind_direction_10m_dominant": "WindDirection"
            }, inplace=True)
            
            df["Date"] = pd.to_datetime(df["time"])
            df.drop(columns=["time"], inplace=True)
            
            df.to_csv(root / "data" / "raw" / f"{name}_weather.csv", index=False)
            print(f"✅ Success: {name} saved.")
        else:
            print(f"❌ Error: Could not extract data for {name}.")
            
    except Exception as e:
        print(f"⚠️ Error for {name}: {e}")
    
    time.sleep(1.5) 

print("\nFinished!")

Processing: MUSKIZ...
✅ Success: MUSKIZ saved.
Processing: SANTURCE...
✅ Success: SANTURCE saved.

Finished!


In [31]:
raw_data_path = root / "data" / "raw"

all_files = list(raw_data_path.glob("*_weather.csv"))

combined_df = pd.DataFrame()

for file_path in all_files:
    station_name = file_path.stem.replace("_weather", "")
    
    df = pd.read_csv(file_path)
    
    df["Station"] = station_name
    
    combined_df = pd.concat([combined_df, df], ignore_index=True)

combined_df = combined_df.sort_values(by=["Date", "Station"])
combined_df.head()

,Temperature,Humidity,Precipitation,WindSpeed,WindDirection,Date,Station
0,6.4,85,0.0,8.0,176,2015-01-01,ALGORTA_BBIZI2
4144,6.7,85,0.0,8.0,176,2015-01-01,BARAKALDO
8288,4.7,82,0.0,8.0,176,2015-01-01,BASAURI
12432,6.8,85,0.0,8.0,176,2015-01-01,ERANDIO
16576,6.8,83,0.0,8.0,176,2015-01-01,MAZARREDO


In [32]:
combined_df.info()

<class 'pandas.DataFrame'>
Index: 33152 entries, 0 to 33151
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Temperature    33152 non-null  float64
 1   Humidity       33152 non-null  int64  
 2   Precipitation  33152 non-null  float64
 3   WindSpeed      33152 non-null  float64
 4   WindDirection  33152 non-null  int64  
 5   Date           33152 non-null  str    
 6   Station        33152 non-null  str    
dtypes: float64(3), int64(2), str(2)
memory usage: 2.6 MB


In [9]:
combined_df.isnull().sum()

Temperature      0
Humidity         0
Precipitation    0
WindSpeed        0
WindDirection    0
Date             0
Station          0
dtype: int64

In [10]:
combined_df.to_csv(root / "data" / "processed" / "weather_data.csv", index=False)

print("✅ Success: Weather data saved.")

✅ Success: Weather data saved.


In [15]:
weather_df = pd.read_csv(root / "data" / "processed" / "weather_data.csv")
air_quality_df = pd.read_csv(root / "data" / "processed" / "cleaned_air_quality_bilbao_2015_2026.csv")
print(weather_df.head())
print(air_quality_df.head())

   Temperature  Humidity  Precipitation  WindSpeed  WindDirection        Date  \
0          6.4        85            0.0        8.0            176  2015-01-01   
1          6.7        85            0.0        8.0            176  2015-01-01   
2          4.7        82            0.0        8.0            176  2015-01-01   
3          6.8        85            0.0        8.0            176  2015-01-01   
4          6.8        83            0.0        8.0            176  2015-01-01   

          Station  
0  ALGORTA_BBIZI2  
1       BARAKALDO  
2         BASAURI  
3         ERANDIO  
4       MAZARREDO  
         Date         station   Town Province   Latitude  Longitude   NO2  \
0  2015-01-01  ALGORTA_BBIZI2  Getxo  Bizkaia  43.362056  -3.022782  47.0   
1  2015-01-02  ALGORTA_BBIZI2  Getxo  Bizkaia  43.362056  -3.022782  56.0   
2  2015-01-03  ALGORTA_BBIZI2  Getxo  Bizkaia  43.362056  -3.022782  48.0   
3  2015-01-04  ALGORTA_BBIZI2  Getxo  Bizkaia  43.362056  -3.022782  43.0   
4  2015-

In [16]:
weather_df["Date"] = pd.to_datetime(weather_df["Date"])
air_quality_df["Date"] = pd.to_datetime(air_quality_df["Date"])

In [26]:
air_quality_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27883 entries, 0 to 27882
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       27883 non-null  datetime64[us]
 1   station    27883 non-null  str           
 2   Town       27883 non-null  str           
 3   Province   27883 non-null  str           
 4   Latitude   27883 non-null  float64       
 5   Longitude  27883 non-null  float64       
 6   NO2        27883 non-null  float64       
 7   PM10       27883 non-null  float64       
 8   PM2.5      27883 non-null  float64       
 9   SO2        27883 non-null  float64       
dtypes: datetime64[us](1), float64(6), str(3)
memory usage: 2.7 MB


In [18]:
merged_df = pd.merge(
    air_quality_df, 
    weather_df, 
    left_on=["Date", "station"],  #  air_quality_df 
    right_on=["Date", "Station"], #  weather_df
    how="left"
)

merged_df = merged_df.drop(columns=["station"])

print(merged_df.head())

        Date   Town Province   Latitude  Longitude   NO2  PM10  PM2.5  SO2  \
0 2015-01-01  Getxo  Bizkaia  43.362056  -3.022782  47.0  25.0   28.0  9.0   
1 2015-01-02  Getxo  Bizkaia  43.362056  -3.022782  56.0  24.0   18.0  8.0   
2 2015-01-03  Getxo  Bizkaia  43.362056  -3.022782  48.0  33.0   21.0  8.0   
3 2015-01-04  Getxo  Bizkaia  43.362056  -3.022782  43.0  31.0   23.0  7.0   
4 2015-01-05  Getxo  Bizkaia  43.362056  -3.022782  29.0  18.0   11.0  5.0   

   Temperature  Humidity  Precipitation  WindSpeed  WindDirection  \
0          6.4      85.0            0.0        8.0          176.0   
1          8.3      81.0            0.0        8.9          207.0   
2          8.9      80.0            0.0       13.0          215.0   
3          9.6      88.0            0.0        8.7          219.0   
4          9.0      85.0            0.0       13.9          170.0   

          Station  
0  ALGORTA_BBIZI2  
1  ALGORTA_BBIZI2  
2  ALGORTA_BBIZI2  
3  ALGORTA_BBIZI2  
4  ALGORTA_BBIZI

In [19]:
# چاپ تعداد ردیف‌ها
print(f"تعداد ردیف‌های فایل کیفیت هوا: {air_quality_df.shape[0]}")
print(f"تعداد ردیف‌های فایل آب و هوا: {weather_df.shape[0]}")
print(f"تعداد ردیف‌های نهایی پس از ادغام: {merged_df.shape[0]}")

# اگر می‌خواهید تعداد ردیف‌ها را به تفکیک هر ایستگاه ببینید:
print("\nتعداد ردیف‌ها در فایل ادغام شده به تفکیک ایستگاه:")
print(merged_df.groupby("Station").size())

تعداد ردیف‌های فایل کیفیت هوا: 27883
تعداد ردیف‌های فایل آب و هوا: 29008
تعداد ردیف‌های نهایی پس از ادغام: 27883

تعداد ردیف‌ها در فایل ادغام شده به تفکیک ایستگاه:
Station
ALGORTA_BBIZI2    3399
BARAKALDO         4144
BASAURI           4144
ERANDIO           4144
MAZARREDO         4144
MUSKIZ            4143
dtype: int64


In [20]:
merged_df.isnull().sum()

Date                0
Town                0
Province            0
Latitude            0
Longitude           0
NO2                 0
PM10                0
PM2.5               0
SO2                 0
Temperature      3765
Humidity         3765
Precipitation    3765
WindSpeed        3765
WindDirection    3765
Station          3765
dtype: int64

In [24]:
print(merged_df[merged_df['Temperature'].isna()]['Date'].dt.year.value_counts())

Date
2016    366
2024    366
2015    365
2019    365
2021    365
2022    365
2023    365
2025    365
2020    364
2018    353
2026    126
Name: count, dtype: int64


In [27]:
# تعداد ردیف‌ها به تفکیک ایستگاه در هر فایل
print("تعداد در فایل آب‌وهوا:")
print(weather_df.groupby('Station').size())

print("\nتعداد در فایل کیفیت هوا:")
print(air_quality_df.groupby('station').size())

تعداد در فایل آب‌وهوا:
Station
ALGORTA_BBIZI2    4144
BARAKALDO         4144
BASAURI           4144
ERANDIO           4144
MAZARREDO         4144
MUSKIZ            4144
SANTURTZI         4144
dtype: int64

تعداد در فایل کیفیت هوا:
station
ALGORTA_BBIZI2    3399
BARAKALDO         4144
BASAURI           4144
ERANDIO           4144
MAZARREDO         4144
MUSKIZ            4143
SANTURCE          3765
dtype: int64


In [39]:
# اطمینان از فرمت تاریخ
air_quality_df['Date'] = pd.to_datetime(air_quality_df['Date'])

# استخراج سال و ماه برای گروه‌بندی
air_quality_df['Year'] = air_quality_df['Date'].dt.year
air_quality_df['Month'] = air_quality_df['Date'].dt.month

# ساخت جدول گزارش به تفکیک ایستگاه، سال و ماه
# این خروجی به شما می‌گوید هر ایستگاه در هر ماه چند بار ثبت شده است
station_monthly_counts = air_quality_df.pivot_table(
    index=['station', 'Year'], 
    columns='Month', 
    values='Date', 
    aggfunc='count', 
    fill_value=0
)

print(station_monthly_counts)

# ذخیره گزارش برای بررسی دقیق‌تر در اکسل
station_monthly_counts.to_csv(root /"reports"/"data_availability_report.csv")

Month                1   2   3   4   5   6   7   8   9   10  11  12
station        Year                                                
ALGORTA_BBIZI2 2015  31  28  31  30  31  30  31  31  30  31  30  31
               2017  31  28  31  30  31  30  31  31  30  31  30  31
               2018  31  28  31  30  31  30  31  31  30  31  30  31
               2020  31  29  31  30  31  30  31  31  30  31  30  31
               2021  31  28  31  30  31  30  31  31  30  31  29  31
...                  ..  ..  ..  ..  ..  ..  ..  ..  ..  ..  ..  ..
SANTURCE       2022  31  28  31  30  31  30  31  31  30  31  30  31
               2023  31  28  31  30  31  30  31  31  30  31  30  31
               2024  31  29  31  30  31  30  31  31  30  31  30  31
               2025  31  28  31  30  31  30  31  31  30  31  30  31
               2026  31  28  31  30   6   0   0   0   0   0   0   0

[81 rows x 12 columns]
